Project: Face Recognition using traditional methods.

Built using LBP, HOG AND GABOR FILTER combined.

The google colab link tested the methods used separately:
https://colab.research.google.com/drive/1TwPOm3aCgkt6S0dmeGscjRJcuWRQKWm3?usp=sharing

The links for the papers used which contains the approach of the methods incorperated in the code:
https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=4803030

https://jceps.utq.edu.iq/index.php/main/article/view/34/22

Data Privacy maintained



In [ ]:
# Imports and defining path

import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import hog, local_binary_pattern
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

TRAIN_DIR = "/content/drive/MyDrive/ECE501_Mugshots/train"
TEST_DIR = "/content/drive/MyDrive/ECE501_Mugshots/test"
TRAIN_LABELS_PATH = "/content/drive/MyDrive/ECE501_Mugshots/train_labels.csv"
SAMPLE_SUBMISSION_PATH = "/content/drive/MyDrive/ECE501_Mugshots/sample_submission.csv"
SUBMISSION_OUTPUT_PATH = "submission.csv"
RANDOM_STATE = 42

Mounted at /content/drive


Reads the training labels file then initializes a *"LabelEncoder()"* to convert string labels into numerical format.



In [ ]:
# Load Labels

df_labels = pd.read_csv(TRAIN_LABELS_PATH)
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)

le = LabelEncoder()
y_full = le.fit_transform(df_labels['label'])
print(f"Loaded {len(df_labels)} samples across {len(le.classes_)} classes")

Loaded 216 samples across 72 classes


Following Cell will include:

*   **augment_image(img):**  Augmentation helps to increase the size and diversity of the training data, which can improve the robustness of a model. The augmentations include horizontal flipping, rotation, changes in brightness and contrast, and adding Gaussian noise.
*   **preprocess_image(img):** It converts the image to grayscale, resizes it to 128x128 pixels, applies histogram equalization to improve contrast, and normalizes the pixel values to the range [0, 1].
*   **extract_gabor_features(img):** Gabor filters are sensitive to edges and textures at different orientations and scales. The function applies Gabor filters with different orientations and calculates the mean and variance of the filtered images, which are used as features.
*  **extract_features(img):** This function extracts a combination of features from the input image, including:
    *  *Histogram of Oriented Gradients (HOG)*: Describes the distribution of edge directions in the image.
    * *Local Binary Pattern (LBP):* Captures local texture information.
    * *Gabor features:* As extracted by the extract_gabor_features function. It concatenates these features into a single feature vector.



*  ** extract_features_augmented(img_path):** This function takes the path to an image file, reads the image, preprocesses it using preprocess_image, applies data augmentation using augment_image, and then extracts features from each augmented image using extract_features. It returns a list of feature vectors, one for each augmented image.




In [ ]:
# Augmentation + Feature Extraction

def augment_image(img):

    aug_images = [img]
    aug_images.append(cv2.flip(img, 1))  # horizontal flip

    # set rotate
    for angle in [-10, 10]:
        M = cv2.getRotationMatrix2D((img.shape[1]//2, img.shape[0]//2), angle, 1)
        aug_images.append(cv2.warpAffine(img, M, (img.shape[1], img.shape[0])))

    # set Brightness
    for alpha in [0.8, 1.2]:
        aug_images.append(cv2.convertScaleAbs(img, alpha=alpha, beta=0) / 255.0)

    # set Contrast
    # CLAHE-Contrast Limited Adaptive Histogram Equalization (convert to 8-bit before applying)
    img_8bit = (img * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_img = clahe.apply(img_8bit)
    aug_images.append(clahe_img / 255.0)  # normalize back to [0,1]

    # Adding Gaussian noise
    noise = np.random.normal(0, 0.02, img.shape)
    noisy = np.clip(img + noise, 0, 1)
    aug_images.append(noisy)

    return aug_images

def preprocess_image(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (128, 128))
    gray = cv2.equalizeHist(gray)
    gray = gray / 255.0
    return gray

def extract_gabor_features(img):
    """Extract Gabor energy features from multiple orientations"""
    gabor_features = []
    for theta in np.arange(0, np.pi, np.pi/4):
        kernel = cv2.getGaborKernel((9, 9), 4.0, theta, 10.0, 0.5, 0, ktype=cv2.CV_32F)
        fimg = cv2.filter2D(img, cv2.CV_8UC3, kernel)
        gabor_features.append(fimg.mean())
        gabor_features.append(fimg.var())
    return np.array(gabor_features)

def extract_features(img):
    """Extract fused HOG + LBP + Gabor features"""
    hog_feat = hog(img, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys')
    lbp = local_binary_pattern((img*255).astype('uint8'), P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 10), range=(0, 9))
    hist = hist.astype("float") / (hist.sum() + 1e-7)
    gabor_feat = extract_gabor_features(img)
    return np.concatenate([hog_feat, hist, gabor_feat])

def extract_features_augmented(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f"Could not read {img_path}")
        return []
    img = preprocess_image(img)
    feats_list = []
    for aug in augment_image(img):
        feats_list.append(extract_features(aug))
    return feats_list

In [ ]:
# Extract Train Features
X = []
y_aug = []
for i, fname in enumerate(df_labels['filename']):
    path = os.path.join(TRAIN_DIR, fname)
    feats = extract_features_augmented(path)
    if feats:
        X.extend(feats)
        y_aug.extend([y_full[i]] * len(feats))

X = np.array(X)
y = np.array(y_aug)
print(f"Features shape: {X.shape}, Classes: {len(np.unique(y))}")

Could not read /content/drive/MyDrive/ECE501_Mugshots/train/ece501_AU2340059_neutral1.jpg
Could not read /content/drive/MyDrive/ECE501_Mugshots/train/ece501_AU2340059_neutral2.jpg
Could not read /content/drive/MyDrive/ECE501_Mugshots/train/ece501_AU2340059_glasses.jpg
Features shape: (1704, 8117), Classes: 71


Next cell will first scales the extracted features using **StandardScaler** to normalize their range. Then, it applies **Principal Component Analysis (PCA)** to reduce the dimensionality of the data while keeping 98% of the variance. This process prepares the features for model training by improving numerical stability and reducing computational load.

In [ ]:
# Scaling + PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(0.98, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA reduced to {X_pca.shape[1]} components")

PCA reduced to 1147 components




*   Splits the data into training and validation sets (X_train, X_val, y_train, y_val) using train_test_split.
* Ensures equal class distribution in splits with stratify=y.
* Defines a grid of hyperparameters (param_grid) to test for the SVM model (C, gamma, kernel).
* Uses GridSearchCV to find the best combination of hyperparameters for an SVC model using 3-fold cross-validation (cv=3).
* Fits the GridSearchCV to the training data (grid.fit).
* Selects the best performing SVM model (svm = grid.best_estimator_).



In [ ]:
# Splitting and Training the Model using SVM with GridSearch

X_train, X_val, y_train, y_val = train_test_split(
    X_pca, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train)}, Val: {len(X_val)}")

param_grid = {
    'C': [1, 5, 10],
    'gamma': ['scale', 0.001, 0.01, 0.1],
    'kernel': ['rbf']
}

grid = GridSearchCV(
    SVC(probability=True),
    param_grid,
    cv=3,
    n_jobs=-1,
    verbose=1,
    scoring='accuracy'
)
grid.fit(X_train, y_train)

svm = grid.best_estimator_
print(f"\nBest SVM Parameters: {grid.best_params_}")


Train: 1363, Val: 341
Fitting 3 folds for each of 12 candidates, totalling 36 fits

Best SVM Parameters: {'C': 5, 'gamma': 'scale', 'kernel': 'rbf'}


In [ ]:
# Test Prediction + Submission

def extract_features_test(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f"Could not read {img_path}")
        return None
    img = preprocess_image(img)
    return extract_features(img)

X_test = []
valid_filenames = []
for fname in sample_sub['filename']:
    path = os.path.join(TEST_DIR, fname)
    feats = extract_features_test(path)
    if feats is not None:
        X_test.append(feats)
        valid_filenames.append(fname)

X_test = np.array(X_test)
X_test_scaled = scaler.transform(X_test)
X_test_pca = pca.transform(X_test_scaled)

y_pred_test = svm.predict(X_test_pca)
y_pred_labels = le.inverse_transform(y_pred_test)

submission = pd.DataFrame({
    'filename': valid_filenames,
    'label': y_pred_labels
})
submission.to_csv(SUBMISSION_OUTPUT_PATH, index=False)
print(f"\n Submission saved: {SUBMISSION_OUTPUT_PATH}")

Could not read /content/drive/MyDrive/ECE501_Mugshots/test/img_00061.jpg
Could not read /content/drive/MyDrive/ECE501_Mugshots/test/img_00062.jpg
Could not read /content/drive/MyDrive/ECE501_Mugshots/test/img_00063.jpg

 Submission saved: submission.csv
